# 04 — Full Pipeline: Train → PGD Attack → MILP Verification

End-to-end baseline pipeline for the thesis:

| Step | What | Thesis WP |
|------|------|-----------|
| 1 | Fix MNIST evaluation split (100 samples, seed 1234) | WP0 |
| 2 | Train MLP (784→128→64→10) and small CNN on MNIST | WP1 |
| 3 | PGD L∞ adversarial attack evaluation on fixed subset | WP1 |
| 4 | MILP exact robustness verification on MLP (5 samples) | WP2 |

> **Tip:** Enable GPU (*Runtime → Change runtime type → GPU*) to speed up training.
> MILP verification always runs on CPU.

In [ ]:
!pip install -q torch torchvision numpy pandas pyyaml tqdm ortools

## 1 — All library code

In [ ]:
from __future__ import annotations

import json
import os
import random
import time
from dataclasses import asdict, dataclass
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn, optim, Tensor
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm import tqdm


# ── Seed ───────────────────────────────────────────────────────────────────────
def set_seed(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)


# ── Data ───────────────────────────────────────────────────────────────────────
def get_mnist_datasets(data_dir="data"):
    tfm = transforms.ToTensor()
    train_ds = datasets.MNIST(str(data_dir), train=True,  download=True, transform=tfm)
    test_ds  = datasets.MNIST(str(data_dir), train=False, download=True, transform=tfm)
    return train_ds, test_ds


def make_loader(dataset, batch_size, shuffle, num_workers, seed):
    gen = torch.Generator()
    gen.manual_seed(int(seed))
    return DataLoader(dataset, batch_size=int(batch_size), shuffle=bool(shuffle),
                      num_workers=int(num_workers), generator=gen, drop_last=False, pin_memory=False)


# ── Evaluation split ───────────────────────────────────────────────────────────
@dataclass(frozen=True)
class Split:
    seed: int
    indices: list[int]


def load_split(path) -> Split:
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    return Split(seed=int(obj["seed"]), indices=[int(i) for i in obj["indices"]])


def ensure_mnist_eval_split(path, seed=1234, n=100) -> Split:
    p = Path(path)
    if p.exists(): return load_split(p)
    indices = random.Random(seed).sample(range(10_000), n)
    split = Split(seed=seed, indices=indices)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps({"seed": split.seed, "indices": split.indices}, indent=2) + "\n", encoding="utf-8")
    return split


# ── Models ─────────────────────────────────────────────────────────────────────
class MnistMlp(nn.Module):
    def __init__(self, in_dim=784, h1=128, h2=64, num_classes=10):
        super().__init__()
        self.fc1  = nn.Linear(int(in_dim), int(h1))
        self.fc2  = nn.Linear(int(h1), int(h2))
        self.fc3  = nn.Linear(int(h2), int(num_classes))
        self.relu = nn.ReLU()

    def forward(self, x):
        if x.ndim == 4: x = x.view(x.shape[0], -1)
        return self.fc3(self.relu(self.fc2(self.relu(self.fc1(x)))))

    def linear_layers(self): return [self.fc1, self.fc2, self.fc3]


class CnnSmall(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(32*7*7, 64), nn.ReLU(), nn.Linear(64, int(num_classes)),
        )

    def forward(self, x): return self.classifier(self.features(x))


# ── Checkpoint I/O ─────────────────────────────────────────────────────────────
@dataclass(frozen=True)
class CheckpointMeta:
    model_type: str
    model_kwargs: dict
    run_name: str


def build_model(model_type, model_kwargs):
    if model_type == "mlp":       return MnistMlp(**model_kwargs)
    if model_type == "cnn_small": return CnnSmall(**model_kwargs)
    raise ValueError(f"Unknown model_type={model_type!r}")


def save_checkpoint(path, model, meta: CheckpointMeta, metrics=None):
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    payload = {"meta": asdict(meta), "model_state_dict": model.state_dict(), "metrics": metrics or {}}
    torch.save(payload, str(p))
    p.with_suffix(".meta.json").write_text(json.dumps(payload["meta"], indent=2) + "\n", encoding="utf-8")


def load_checkpoint(path, map_location="cpu"):
    payload = torch.load(str(path), map_location=map_location)
    raw = payload["meta"]
    meta = CheckpointMeta(
        model_type=str(raw["model_type"]),
        model_kwargs=dict(raw.get("model_kwargs", {})),
        run_name=str(raw.get("run_name", "run")),
    )
    model = build_model(meta.model_type, meta.model_kwargs)
    model.load_state_dict(payload["model_state_dict"])
    return model, meta, payload.get("metrics", {})


# ── PGD attack ─────────────────────────────────────────────────────────────────
def pgd_linf(model, x0, y, eps, steps, step_size, random_start=False):
    model.eval()
    x0 = x0.detach()
    x  = x0.clone()
    if random_start:
        lo = (x0 - float(eps)).clamp(0.0, 1.0)
        hi = (x0 + float(eps)).clamp(0.0, 1.0)
        x  = (x + (2 * torch.rand_like(x) - 1) * float(eps)).clamp(lo, hi)
    for _ in range(int(steps)):
        x.requires_grad_(True)
        loss = F.cross_entropy(model(x), y, reduction="sum")
        grad = torch.autograd.grad(loss, x)[0]
        with torch.no_grad():
            lo = (x0 - float(eps)).clamp(0.0, 1.0)
            hi = (x0 + float(eps)).clamp(0.0, 1.0)
            x  = (x + float(step_size) * grad.sign()).clamp(lo, hi)
        x = x.detach()
    return x


# ── MILP types ─────────────────────────────────────────────────────────────────
class VerificationStatus(str, Enum):
    VERIFIED  = "VERIFIED"
    FALSIFIED = "FALSIFIED"
    TIMEOUT   = "TIMEOUT"
    ERROR     = "ERROR"


@dataclass
class VerificationResult:
    status: VerificationStatus
    worst_margin: float | None
    solve_time: float | None
    solver_name: str
    adv_example: list[float] | None


@dataclass
class LinExpr:
    terms: Dict[Any, float]
    const: float = 0.0


def expr_const(c):         return LinExpr(terms={}, const=float(c))
def expr_var(v, coeff=1.): return LinExpr(terms={v: float(coeff)}, const=0.0)
def expr_mul(a, s):        return LinExpr(terms={v: c*float(s) for v, c in a.terms.items()}, const=a.const*float(s))

def expr_add(a, b):
    t = dict(a.terms)
    for v, c in b.terms.items(): t[v] = t.get(v, 0.0) + c
    return LinExpr(terms=t, const=a.const + b.const)

def expr_sub(a, b): return expr_add(a, expr_mul(b, -1.0))


# ── IBP bounds ─────────────────────────────────────────────────────────────────
def compute_mlp_ibp_bounds(model, x_center, eps):
    x0  = x_center.detach().view(-1)
    x_l = torch.clamp(x0 - float(eps), 0.0, 1.0)
    x_u = torch.clamp(x0 + float(eps), 0.0, 1.0)
    h_l, h_u = [x_l], [x_u]
    a_l_list, a_u_list = [], []
    linears = list(model.linear_layers())
    for li, layer in enumerate(linears):
        W  = layer.weight.detach()
        b  = layer.bias.detach()
        Wp = torch.clamp(W, min=0.0)
        Wn = torch.clamp(W, max=0.0)
        al = Wp @ h_l[-1] + Wn @ h_u[-1] + b
        au = Wp @ h_u[-1] + Wn @ h_l[-1] + b
        a_l_list.append(al)
        a_u_list.append(au)
        if li < len(linears) - 1:
            h_l.append(torch.relu(al))
            h_u.append(torch.relu(au))
    return {"x_l": x_l, "x_u": x_u, "a_l_list": a_l_list, "a_u_list": a_u_list}


# ── OR-Tools CBC backend ───────────────────────────────────────────────────────
class OrtoolsCbcBackend:
    solver_name = "CBC"

    def __init__(self):
        from ortools.linear_solver import pywraplp
        self._pw = pywraplp
        s = pywraplp.Solver.CreateSolver("CBC") or pywraplp.Solver.CreateSolver("CBC_MIXED_INTEGER_PROGRAMMING")
        if s is None: raise RuntimeError("Cannot create CBC solver")
        self.solver = s
        self._obj = None

    def add_var(self, lb, ub, binary=False):
        return self.solver.IntVar(0., 1., "") if binary else self.solver.NumVar(float(lb), float(ub), "")

    def add_binary(self): return self.add_var(0, 1, True)

    def _e(self, e):
        lin = self.solver.Sum([c * v for v, c in e.terms.items()])
        return lin + float(e.const) if e.const else lin

    def add_le(self, l, r): self.solver.Add(self._e(expr_sub(l, r)) <= 0.)
    def add_ge(self, l, r): self.solver.Add(self._e(expr_sub(l, r)) >= 0.)
    def add_eq(self, l, r): self.solver.Add(self._e(expr_sub(l, r)) == 0.)

    def set_objective_max(self, e):
        self._obj = e
        self.solver.Maximize(self._e(e))

    def solve(self, time_limit_s=None):
        if time_limit_s is not None: self.solver.set_time_limit(int(max(time_limit_s, 0.) * 1000))
        t0 = time.time()
        st = self.solver.Solve()
        elapsed = time.time() - t0
        pw = self._pw
        sm = {pw.Solver.OPTIMAL: "OPTIMAL", pw.Solver.FEASIBLE: "FEASIBLE",
              pw.Solver.INFEASIBLE: "INFEASIBLE", pw.Solver.UNBOUNDED: "UNBOUNDED"}
        s_str = sm.get(st, "UNKNOWN")
        obj = float(self.solver.Objective().Value()) if self._obj and st in (pw.Solver.OPTIMAL, pw.Solver.FEASIBLE) else None
        return s_str, obj, elapsed

    def get_value(self, v): return float(v.solution_value())


def get_backend(name="auto"):
    lname = name.lower()
    if lname == "cbc": return OrtoolsCbcBackend()
    if lname == "auto":
        try:
            import gurobipy as gp
            m = gp.Model(); m.Params.OutputFlag = 0
            class _GB:
                solver_name = "Gurobi"
                def __init__(self): self.gp=gp; self.model=m; self._obj=None
                def add_var(s,lb,ub,binary=False):
                    return s.model.addVar(lb=float(lb),ub=float(ub),
                                         vtype=gp.GRB.BINARY if binary else gp.GRB.CONTINUOUS)
                def add_binary(s): return s.add_var(0,1,True)
                def _e(s,e):
                    lin=gp.LinExpr()
                    for v,c in e.terms.items(): lin.add(v,float(c))
                    if e.const: lin+=float(e.const)
                    return lin
                def add_le(s,l,r): s.model.addLConstr(s._e(expr_sub(l,r))<=0.)
                def add_ge(s,l,r): s.model.addLConstr(s._e(expr_sub(l,r))>=0.)
                def add_eq(s,l,r): s.model.addLConstr(s._e(expr_sub(l,r))==0.)
                def set_objective_max(s,e): s._obj=e; s.model.setObjective(s._e(e),gp.GRB.MAXIMIZE)
                def solve(s,time_limit_s=None):
                    if time_limit_s: s.model.Params.TimeLimit=float(time_limit_s)
                    t0=time.time(); s.model.optimize(); el=time.time()-t0
                    sm={gp.GRB.OPTIMAL:"OPTIMAL",gp.GRB.SUBOPTIMAL:"FEASIBLE",
                        gp.GRB.TIME_LIMIT:"TIME_LIMIT",gp.GRB.INFEASIBLE:"INFEASIBLE",gp.GRB.UNBOUNDED:"UNBOUNDED"}
                    obj=float(s.model.objVal) if s.model.SolCount>0 else None
                    return sm.get(s.model.Status,"UNKNOWN"),obj,el
                def get_value(s,v): return float(v.X)
            return _GB()
        except Exception:
            return OrtoolsCbcBackend()
    return OrtoolsCbcBackend()


# ── MILP encoding ──────────────────────────────────────────────────────────────
def encode_mlp_on_box(backend, model, x_center, eps):
    bounds = compute_mlp_ibp_bounds(model, x_center, eps)
    x_l, x_u = bounds["x_l"], bounds["x_u"]
    x0 = x_center.detach().view(-1)
    in_vars = [backend.add_var(float(x_l[i]), float(x_u[i])) for i in range(x0.numel())]
    linears = list(model.linear_layers())
    prev = in_vars
    logit_vars = []
    for li, layer in enumerate(linears):
        W, b = layer.weight.detach(), layer.bias.detach()
        a_vars = [backend.add_var(-1e9, 1e9) for _ in range(W.shape[0])]
        for i in range(W.shape[0]):
            rhs = expr_const(float(b[i]))
            for j in range(W.shape[1]):
                c = float(W[i, j])
                if c != 0.: rhs = expr_add(rhs, expr_mul(expr_var(prev[j]), c))
            backend.add_eq(expr_var(a_vars[i]), rhs)
        if li == len(linears) - 1:
            logit_vars = a_vars
        else:
            al, au = bounds["a_l_list"][li], bounds["a_u_list"][li]
            h_vars = []
            for i, av in enumerate(a_vars):
                l, u = float(al[i]), float(au[i])
                if u <= 0.:
                    h = backend.add_var(0., 0.)
                elif l >= 0.:
                    h = backend.add_var(l, u)
                    backend.add_eq(expr_var(h), expr_var(av))
                else:
                    h = backend.add_var(0., u)
                    bb = backend.add_binary()
                    backend.add_ge(expr_var(h), expr_const(0.))
                    backend.add_ge(expr_var(h), expr_var(av))
                    omb = expr_add(expr_const(1.), expr_mul(expr_var(bb), -1.))
                    backend.add_le(expr_var(h), expr_add(expr_var(av), expr_mul(omb, -l)))
                    backend.add_le(expr_var(h), expr_mul(expr_var(bb), u))
                    backend.add_ge(expr_var(av), expr_const(l))
                    backend.add_le(expr_var(av), expr_const(u))
                h_vars.append(h)
            prev = h_vars
    return {"input_vars": in_vars, "logit_vars": logit_vars}


def verify_point(model, x, y, eps, backend_name="auto", time_limit_s=None):
    device = next(model.parameters()).device
    x = x.detach().to(device)
    worst = float("-inf")
    best_adv = None
    t_acc = 0.
    last_solver = ""
    for c in [ci for ci in range(10) if ci != int(y)]:
        bk = get_backend(backend_name)
        enc = encode_mlp_on_box(bk, model, x, eps)
        margin = expr_sub(expr_var(enc["logit_vars"][c]), expr_var(enc["logit_vars"][int(y)]))
        bk.set_objective_max(margin)
        ss, obj, t = bk.solve(time_limit_s=time_limit_s)
        last_solver = bk.solver_name
        t_acc += float(t)
        if ss in {"OPTIMAL", "FEASIBLE"}:    status = VerificationStatus.FALSIFIED
        elif ss in {"INFEASIBLE","UNBOUNDED"}: status = VerificationStatus.VERIFIED
        elif ss == "TIME_LIMIT":             status = VerificationStatus.TIMEOUT
        else:                                 status = VerificationStatus.ERROR
        if status in {VerificationStatus.ERROR, VerificationStatus.TIMEOUT}:
            return VerificationResult(status=status, worst_margin=None,
                                      solve_time=t_acc, solver_name=last_solver, adv_example=None)
        if obj is not None and obj > worst:
            worst = float(obj)
            if status == VerificationStatus.FALSIFIED:
                best_adv = [bk.get_value(v) for v in enc["input_vars"]]
    if worst <= 0.:
        return VerificationResult(status=VerificationStatus.VERIFIED, worst_margin=worst,
                                  solve_time=t_acc, solver_name=last_solver, adv_example=None)
    return VerificationResult(status=VerificationStatus.FALSIFIED, worst_margin=worst,
                              solve_time=t_acc, solver_name=last_solver, adv_example=best_adv)


print("All library code loaded")

## 2 — Pipeline configuration

Edit the values here to change the experiment settings.

In [ ]:
# ── Global ─────────────────────────────────────────────────────────────────────
SEED        = 1234
DATA_DIR    = "data"
SUBSET_PATH = "assets/splits/mnist_eval_100.json"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

# ── Training ───────────────────────────────────────────────────────────────────
MLP_RUN_NAME   = "mlp_mnist"
CNN_RUN_NAME   = "cnn_small_mnist"
BATCH_SIZE     = 128
EPOCHS         = 3
LR             = 1e-3
WEIGHT_DECAY   = 0.0
NUM_WORKERS    = 0

# ── PGD ────────────────────────────────────────────────────────────────────────
PGD_EPS          = 0.03
PGD_STEPS        = 40
PGD_STEP_SIZE    = 0.01
PGD_RANDOM_START = True
PGD_BATCH_SIZE   = 64

# ── MILP ───────────────────────────────────────────────────────────────────────
MILP_EPS         = 0.03
MILP_SOLVER      = "auto"    # "auto", "cbc", "gurobi"
MILP_TIME_LIMIT  = 30.0      # seconds per LP
MILP_MAX_SAMPLES = 5         # keep small — MILP is slow
# ──────────────────────────────────────────────────────────────────────────────

set_seed(SEED)
print(f"Device: {DEVICE} | Seed: {SEED}")

## Step 0 — Fix MNIST evaluation split (WP0)

In [ ]:
split = ensure_mnist_eval_split(SUBSET_PATH, seed=SEED, n=100)
print(f"Evaluation split ready: {len(split.indices)} samples  (seed={split.seed})")
print(f"First 10 indices: {split.indices[:10]}")

## Step 1a — Train MLP (WP1)

In [ ]:
@torch.no_grad()
def eval_accuracy(model, loader, device):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += int((model(x).argmax(1) == y).sum())
        total   += int(y.numel())
    return correct / max(total, 1)


def train_model(model_type, model_kwargs, run_name, epochs=EPOCHS):
    ckpt_path = Path("runs") / run_name / "model.pt"
    if ckpt_path.exists():
        print(f"Checkpoint already exists: {ckpt_path} — skipping training")
        return ckpt_path

    train_ds, test_ds = get_mnist_datasets(DATA_DIR)
    train_loader = make_loader(train_ds, BATCH_SIZE, True,  NUM_WORKERS, SEED)
    test_loader  = make_loader(test_ds,  BATCH_SIZE, False, NUM_WORKERS, SEED)
    model = build_model(model_type, model_kwargs).to(DEVICE)
    opt   = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    metrics = {"epochs": []}
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = correct = total = 0
        start = time.time()
        for x, y in tqdm(train_loader, desc=f"{run_name} epoch {epoch}/{epochs}", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss   = F.cross_entropy(logits, y)
            loss.backward()
            opt.step()
            epoch_loss += float(loss.item()) * y.numel()
            correct    += int((logits.argmax(1) == y).sum())
            total      += int(y.numel())
        train_loss = epoch_loss / max(total, 1)
        train_acc  = correct / max(total, 1)
        test_acc   = eval_accuracy(model, test_loader, DEVICE)
        metrics["epochs"].append({"epoch": epoch, "train_loss": train_loss,
                                   "train_acc": train_acc, "test_acc": test_acc})
        print(f"  epoch={epoch}  train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  test_acc={test_acc:.4f}")

    metrics["total_time_s"] = time.time() - t0
    meta = CheckpointMeta(model_type=model_type, model_kwargs=model_kwargs, run_name=run_name)
    save_checkpoint(ckpt_path, model, meta=meta, metrics=metrics)
    (Path("runs") / run_name / "metrics.json").write_text(
        json.dumps(metrics, indent=2) + "\n", encoding="utf-8"
    )
    print(f"  Saved: {ckpt_path}")
    return ckpt_path


print("Training MLP...")
MLP_KWARGS = {"in_dim": 784, "h1": 128, "h2": 64, "num_classes": 10}
mlp_ckpt   = train_model("mlp", MLP_KWARGS, MLP_RUN_NAME)

## Step 1b — Train small CNN (WP1)

In [ ]:
print("Training CNN...")
CNN_KWARGS = {"num_classes": 10}
cnn_ckpt   = train_model("cnn_small", CNN_KWARGS, CNN_RUN_NAME)

## Step 2 — PGD adversarial evaluation on MLP (WP1)

In [ ]:
print("Running PGD evaluation on MLP...")
model_pgd, meta_pgd, _ = load_checkpoint(mlp_ckpt, map_location=DEVICE)
model_pgd.to(DEVICE).eval()

_, test_ds_pgd = get_mnist_datasets(DATA_DIR)
sub_ds_pgd = Subset(test_ds_pgd, split.indices)
loader_pgd = DataLoader(sub_ds_pgd, batch_size=PGD_BATCH_SIZE, shuffle=False, num_workers=0)

rows_pgd = []
t0_pgd   = time.time()

for batch_idx, (x0, y) in enumerate(loader_pgd):
    x0, y = x0.to(DEVICE), y.to(DEVICE)
    with torch.no_grad():
        clean_pred = model_pgd(x0).argmax(1)
        clean_loss = F.cross_entropy(model_pgd(x0), y, reduction="none")
    x_adv = pgd_linf(model_pgd, x0, y, PGD_EPS, PGD_STEPS, PGD_STEP_SIZE, PGD_RANDOM_START)
    with torch.no_grad():
        adv_pred = model_pgd(x_adv).argmax(1)
        adv_loss = F.cross_entropy(model_pgd(x_adv), y, reduction="none")
    base = batch_idx * PGD_BATCH_SIZE
    for i in range(y.shape[0]):
        rows_pgd.append({
            "index": int(split.indices[base + i]), "y": int(y[i]),
            "clean_pred": int(clean_pred[i]), "adv_pred": int(adv_pred[i]),
            "success": int(adv_pred[i]) != int(y[i]),
            "clean_loss": float(clean_loss[i]), "adv_loss": float(adv_loss[i]),
        })

df_pgd = pd.DataFrame(rows_pgd)
Path("results").mkdir(parents=True, exist_ok=True)
pgd_out = f"results/pgd_{meta_pgd.run_name}_eps{PGD_EPS:.4f}.csv"
df_pgd.to_csv(pgd_out, index=False)

asr        = df_pgd["success"].mean()
clean_acc  = (df_pgd["clean_pred"] == df_pgd["y"]).mean()
print(f"Clean accuracy on subset : {clean_acc:.4f}")
print(f"PGD attack success rate  : {asr:.4f} ({df_pgd['success'].sum()}/{len(df_pgd)})")
print(f"Results saved to         : {pgd_out}  [{time.time()-t0_pgd:.1f}s]")

## Step 3 — MILP exact verification on MLP (WP2)

This is slow. `MILP_MAX_SAMPLES = 5` by default — increase carefully.

In [ ]:
print("Running MILP verification on MLP...")
model_milp, meta_milp, _ = load_checkpoint(mlp_ckpt, map_location="cpu")
model_milp.eval()

_, test_ds_milp = get_mnist_datasets(DATA_DIR)
milp_indices = split.indices[:MILP_MAX_SAMPLES]
sub_ds_milp  = Subset(test_ds_milp, milp_indices)
loader_milp  = DataLoader(sub_ds_milp, batch_size=1, shuffle=False, num_workers=0)

rows_milp = []
adv_dir   = Path("results/adversarial")
adv_dir.mkdir(parents=True, exist_ok=True)

for local_idx, (x, y) in enumerate(loader_milp):
    idx   = int(milp_indices[local_idx])
    label = int(y[0])
    print(f"  [{local_idx+1}/{len(milp_indices)}] idx={idx} label={label} ...", end=" ", flush=True)

    res = verify_point(model_milp, x[0], label, MILP_EPS,
                       backend_name=MILP_SOLVER, time_limit_s=MILP_TIME_LIMIT)
    print(f"{res.status.value}  margin={res.worst_margin}  t={res.solve_time:.2f}s  solver={res.solver_name}")

    adv_path_str = None
    if res.status == VerificationStatus.FALSIFIED and res.adv_example is not None:
        adv      = np.array(res.adv_example, dtype=np.float32).reshape(1, 28, 28)
        adv_path = adv_dir / f"mnist_idx{idx}_eps{MILP_EPS:.4f}.npz"
        np.savez_compressed(adv_path, x_adv=adv, x0=x.numpy(), eps=float(MILP_EPS), y=int(label))
        adv_path_str = str(adv_path)

    rows_milp.append({
        "index": idx, "y": label,
        "status": res.status.value, "worst_margin": res.worst_margin,
        "time_s": res.solve_time, "solver": res.solver_name, "adv_path": adv_path_str,
    })

df_milp  = pd.DataFrame(rows_milp)
milp_out = f"results/milp_{meta_milp.run_name}_eps{MILP_EPS:.4f}.csv"
df_milp.to_csv(milp_out, index=False)
print(f"\nResults saved to: {milp_out}")

## Final Summary

In [ ]:
print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"\n[WP0] Eval split: {len(split.indices)} samples (seed={split.seed})")
print(f"\n[WP1] PGD (eps={PGD_EPS}, steps={PGD_STEPS}):")
print(f"      Clean accuracy : {(df_pgd['clean_pred']==df_pgd['y']).mean():.4f}")
print(f"      Attack success : {df_pgd['success'].mean():.4f}")
print(f"\n[WP2] MILP verification (eps={MILP_EPS}, solver={MILP_SOLVER}):")
print(df_milp[["index","y","status","worst_margin","time_s"]].to_string(index=False))
print()
print("Status counts:", dict(df_milp["status"].value_counts()))